In [8]:
#RGB TO GREYSCALE and the time diff

import numpy as np
import torch
import time

# Create 4K synthetic image
H, W = 4096, 4096
img = np.random.randint(0, 256, (H, W, 3), dtype=np.uint8)

# CPU timing (NumPy)
start_cpu = time.time()
gray_cpu = 0.299*img[:,:,2] + 0.587*img[:,:,1] + 0.114*img[:,:,0]
end_cpu = time.time()

cpu_time = end_cpu - start_cpu
print("CPU Time:", cpu_time, "seconds")

# GPU timing (PyTorch)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

img_gpu = torch.tensor(img, dtype=torch.float32).to(device)

# make sure GPU is ready
if device.type == "cuda":
    torch.cuda.synchronize()

start_gpu = time.time()

gray_gpu = 0.299*img_gpu[:,:,2] + 0.587*img_gpu[:,:,1] + 0.114*img_gpu[:,:,0]

if device.type == "cuda":
    torch.cuda.synchronize()

end_gpu = time.time()

gpu_time = end_gpu - start_gpu
print("GPU Time:", gpu_time, "seconds")

# Time Difference
print("Time Difference (CPU - GPU):", cpu_time - gpu_time, "seconds")

CPU Time: 0.17919087409973145 seconds
Device: cuda
GPU Time: 0.10481691360473633 seconds
Time Difference (CPU - GPU): 0.07437396049499512 seconds


In [9]:
try:
    print("Starting test...")

    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using:", device)

    class MyCNN(nn.Module):
        def __init__(self):
            super(MyCNN, self).__init__()
            self.conv1 = nn.Conv2d(3, 16, 3, 1, 1)
            self.pool = nn.MaxPool2d(2, 2)
            self.fc1 = nn.LazyLinear(1)



        def forward(self, x):
            x = self.pool(F.relu(self.conv1(x)))
            x = torch.flatten(x, 1)
            x = torch.sigmoid(self.fc1(x))
            print(x.shape)

            return x

    model = MyCNN().to(device)
    print("Model created")

    x = torch.randn(1, 3, 128, 128).to(device)
    out = model(x)

    print("Output shape:", out.shape)

except Exception as e:
    print("Error:", e)

Starting test...
Using: cuda
Model created
torch.Size([1, 1])
Output shape: torch.Size([1, 1])
